# Reference HCO & Affiliations Crosswalk

## Version Control
| Version        | Description |
|----------------|-------------|
| v4 (1/8)     | Fixed on HCO multiple name issue |
| v3 (12/29)     | Added flag for hco_npi (Present in Julie's file then 1 else 0), Added hcp_source flag, hco source flag |
| v2 (12/24)     | Reference file fully refreshed: Komodo-based HCP–HCO affiliation backfill, secondary→primary HCO NPI mapping using Julie’s file, HCO zip refresh, territory/region reassignment, and final active HCO column standardization |
| v1 (12/12)     | Baseline reference file with existing HCP–HCO mappings and territory assignments |


**Changing the reference file naming**
- reference_file_12_12_2025 --> base_reference_file
- reference_file_12_24_2025 --> reference_file

In [0]:
WITH

/* STEP 1: HCP input */
hcp_input AS (
    SELECT '1407211667' AS hcp_npi
),

/* STEP 2: Get HCP VID from VOD using HCP NPI */
hcp_vid AS (
    SELECT
        a.hcp_npi,
        b.vid__v AS hcp_vid
    FROM hcp_input a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

/* STEP 3: Pull active HCP–HCO affiliations from VOD, ranked by recency */
ranked_vod_affiliations AS (
    SELECT
        a.hcp_npi,
        c.vid__v          AS hco_vid,
        c.corporate_name__v AS hco_name,
        c.npi_num__v      AS hco_npi,
        b.modified_date__v,
        b.status_update_time__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY
                b.modified_date__v       DESC NULLS LAST,
                b.status_update_time__v  DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v  = 'HCP_HCO'
       and b.entity_type__v = 'HCP'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v   = 'A'
      AND b.relationship_type__v   = '7356'
),

/* STEP 4: Keep only the most recent affiliation */
best_affiliation AS (
    SELECT
        hcp_npi,
        hco_npi,
        hco_name
    FROM ranked_vod_affiliations
    WHERE rn = 1
),

/* STEP 5: Get latest valid HCO ZIP from VOD for that HCO */
hco_zip_ranked AS (
    SELECT
        a.npi_num__v        AS hco_npi,
        b.address_line_1__v AS hco_address,
        b.postal_code_cda__v AS hco_zip,
        ROW_NUMBER() OVER (
            PARTITION BY a.npi_num__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_edp_prd.com_raw.vod_hco a
    JOIN com_edp_prd.com_raw.vod_address b
        ON b.entity_vid__v                  = a.vid__v
       AND b.entity_type__v                 = 'HCO'
       AND b.record_state__v                = 'VALID'
       AND b.address_status__v             IN ('A', 'DS')
       AND b.address_verification_status__v NOT IN ('NS', 'U')
    WHERE a.npi_num__v IN (SELECT hco_npi FROM best_affiliation WHERE hco_npi IS NOT NULL)
),

hco_zip_final AS (
    SELECT hco_npi, hco_address, hco_zip
    FROM hco_zip_ranked
    WHERE rn = 1
)

/* FINAL OUTPUT */
SELECT
    b.hcp_npi,
    b.hco_npi,
    b.hco_name,
    COALESCE(z.hco_address, '-') AS hco_address,
    COALESCE(z.hco_zip,     '-') AS hco_zip
FROM best_affiliation b
LEFT JOIN hco_zip_final z
    ON b.hco_npi = z.hco_npi;

In [0]:
select *
from com_raw.vod_address
where entity_vid__v in ('242977496344036352', '944827502886719071')

In [0]:
select * from com_raw.vod_hco
where corporate_name__v in ('Childrens Health Specialty Center Dallas Campus')

In [0]:
select * from com_edp_prd.com_raw.kom_providers
where npi in ('1700961620')

In [0]:
-- create or replace table com_edp_prd.cmpa_insights_internal_schema.base_reference_file as 
-- select * from com_edp_prd.cmpa_insights_internal_schema.reference_file_12_12_2025

In [0]:
SELECT COUNT(DISTINCT hco_target) 
FROM com_edp_prd.cmpa_insights_internal_schema.base_reference_file
WHERE hco_target != '-';

In [0]:
select count(distinct hco_target) from com_edp_prd.cmpa_insights_internal_schema.reference_file;

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.base_reference_file;

### Secondary to Primary NPI (Intermediary Table)
Stored in Databricks as secondary_to_primary_npi in CMPA schema

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi as
with base_table as (
  select * from com_edp_prd.com_raw.vod_npi where entity_type = 'HCO'
),
active_id as (
  select distinct vid__v, id as active_id
from base_table
where is_displayed = true
),
id_and_active_id as (
  select distinct a.vid__v, a.id as secondary_npi, b.active_id as primary_npi
  from base_table as a
  left join active_id as b
  on a.vid__v = b.vid__v
),
final_output as (
  select vid__v, secondary_npi, primary_npi
from id_and_active_id
where secondary_npi != '-'
order by primary_npi, secondary_npi
),
final_output_with_name as (
  select a.*, b.corporate_name__v as primary_name
  from final_output as a
  left join com_raw.vod_hco as b on a.primary_npi = b.npi_num__v
)
select * from final_output_with_name

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi limit 4

In [0]:
select *
from cmpa_insights_internal_schema.secondary_to_primary_npi
where secondary_npi in ('1144211301','1184779332','1851458038','1225259039','1831318856','1356496772','1003947599','1205822236')

In [0]:
create or replace temporary view npi_mapping as
WITH npi_mapping AS (
    SELECT
        current_npi,
        current_name,
        mapped_npi,
        mapped_name
    FROM (
        VALUES
            ('1144211301', 'Atrium Health Wake Forest Baptist Medical Center', '1295789907', 'Atrium Health'),
            ('1184779332', 'Childrens Healthcare Of Atlanta Scottish Rite Hospital', '1235339227', 'Emory University Hospital'),
            ('1851458038', 'Dr Patrick Leavey MD Office', '1235582925', 'UT Health'),
            ('1225259039', 'Greenwood Genetics Center Inc.', '1649261462', 'Greenwood Genetics Center Inc.'),
            ('1831318856', 'Greenwood Genetics Center Inc.', '1649261462', 'Greenwood Genetics Center Inc.'),
            ('1356496772', 'Kaiser Permanente Fontana Medical Center', '1013062769', 'Kaiser Permanente San Diego Medical Center'),
            ('1003947599', 'Univ. Pediatric Associates Inc.', '1144266024', 'Indiana University Health'),
            ('1205822236', 'Yale Medicine', '1013924182', 'Yale-New Haven Hospital')
    ) AS t(current_npi, current_name, mapped_npi, mapped_name)
)
select * from npi_mapping

In [0]:
select a.*, b.primary_npi, b.primary_name
from npi_mapping as a
left join cmpa_insights_internal_schema.secondary_to_primary_npi as b on a.current_npi = b.secondary_npi

In [0]:
/* ============================================================================
   PURPOSE
   ----------------------------------------------------------------------------
   This script builds a final HCP–HCO reference view by:
   1) Starting from a base reference file
   2) Keeping records that already have an HCO NPI
   3) Deriving missing HCO affiliations using VOD (Salesforce) via HCP→HCO
   4) Backfilling remaining missing affiliations using Komodo
   5) Applying manual NPI remapping (Jess mapping logic)
   6) Mapping secondary NPIs to primary NPIs (Julie mapping logic)
   7) Flagging target HCOs based on an approved NPI list (plus primary-mapped NPIs)
   8) Enriching HCO ZIPs using VOD first, then Komodo as fallback
   9) Reassigning territory & region based on ZIP (HCO ZIP preferred, else HCP ZIP)
  10) Producing a final, clean HCP–HCO reference output

   NOTES
   ----------------------------------------------------------------------------
   - Documentation below is aligned to the *actual* flow of CTEs in the query.
   - No SQL logic has been changed; only comments/step labels were corrected.
   ============================================================================ */

CREATE OR REPLACE TEMPORARY VIEW reference_file_v1 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Load base reference file
   --------------------------------------------------------------------------- */
base_table AS (
    SELECT *
    FROM cmpa_insights_internal_schema.base_reference_file
),

/* ---------------------------------------------------------------------------
   STEP 1: Split records by presence of HCO NPI in the base file
   - v1: Records that already have an HCO NPI
   - v2: Records missing HCO NPI but having HCP NPI (eligible for affiliation search)
   --------------------------------------------------------------------------- */
vod_search_base_table_v1 AS (
    SELECT *
    FROM base_table
    WHERE hco_npi != '-'
),

vod_search_base_table_v2 AS (
    SELECT *
    FROM base_table
    WHERE hco_npi = '-'
      AND hcp_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 2: Fetch HCP VID from VOD using HCP NPI (needed to traverse HCP→HCO)
   --------------------------------------------------------------------------- */
hcp_vid AS (
    SELECT
        a.*,
        b.vid__v AS hcp_vid
    FROM vod_search_base_table_v2 a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

/* ---------------------------------------------------------------------------
   STEP 3: Pull active HCP–HCO affiliations from VOD and rank by recency
   - Filters: active parent HCO status + relationship type
   - Ranking: most recent modified/status update timestamp wins
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v1 AS (
    SELECT
        a.hcp_npi,
        c.vid__v AS hco_vid,
        c.corporate_name__v AS hco_name,
        c.npi_num__v AS hco_npi,
        b.modified_date__v,
        b.status_update_time__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

/* ---------------------------------------------------------------------------
   STEP 4: Select most recent active affiliation per HCP from VOD (rn=1)
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v2 AS (
    SELECT
        hcp_npi,
        hco_npi,
        hco_name
    FROM ranked_vod_affiliations_v1
    WHERE rn = 1
      AND hco_npi IS NOT NULL
),

/* ---------------------------------------------------------------------------
   STEP 5: Apply VOD-derived HCOs to base records missing HCO NPI (from Step 1 v2)
   --------------------------------------------------------------------------- */
vod_affiliations_implementation AS (
    SELECT
        a.hcp_npi,
        a.hcp_target,
        a.hcp_first_name,
        a.hcp_last_name,
        a.hcp_zipcode,
        COALESCE(b.hco_npi, '-') AS hco_npi,
        a.hco_target,
        COALESCE(b.hco_name, '-') AS hco_name,
        a.hco_zip,
        a.territory,
        a.region
    FROM vod_search_base_table_v2 a
    LEFT JOIN ranked_vod_affiliations_v2 b
        ON a.hcp_npi = b.hcp_npi
),

/* ---------------------------------------------------------------------------
   STEP 6: Combine:
   - original records that already had HCO NPI (Step 1 v1)
   - records enriched with VOD-derived HCO NPI/name (Step 5)
   --------------------------------------------------------------------------- */
union_after_vod_check AS (
    SELECT * FROM vod_search_base_table_v1
    UNION
    SELECT * FROM vod_affiliations_implementation
),

/* ---------------------------------------------------------------------------
   STEP 7: Split into records still missing HCO affiliations vs. those with HCO
   --------------------------------------------------------------------------- */
hcp_with_no_affiliations AS (
    SELECT *
    FROM union_after_vod_check
    WHERE hco_npi = '-'
),

hcp_with_affiliations AS (
    SELECT *
    FROM union_after_vod_check
    WHERE hco_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 8: Backfill missing HCOs using Komodo affiliations
   - For INDIVIDUAL HCP NPI: use kom_providers.hco_primary_npi
   - Join to ORGANIZATION record to get HCO name
   --------------------------------------------------------------------------- */
pulling_affiliations_from_komodo AS (
    SELECT
        a.hcp_npi,
        a.hcp_target,
        a.hcp_first_name,
        a.hcp_last_name,
        a.hcp_zipcode,
        COALESCE(b.hco_primary_npi, '-') AS hco_npi,
        a.hco_target,
        COALESCE(c.organization_name, '-') AS hco_name,
        a.hco_zip,
        a.territory,
        a.region
    FROM hcp_with_no_affiliations a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.hcp_npi = b.npi
       AND b.provider_type = 'INDIVIDUAL'
    LEFT JOIN com_edp_prd.com_raw.kom_providers c
        ON b.hco_primary_npi = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ---------------------------------------------------------------------------
   STEP 9: Combine all affiliations (VOD + Komodo)
   --------------------------------------------------------------------------- */
finalizing_affiliations AS (
    SELECT * EXCEPT(hco_target)
    FROM (
        SELECT * FROM hcp_with_affiliations
        UNION
        SELECT * FROM pulling_affiliations_from_komodo
    )
),

/* ---------------------------------------------------------------------------
   STEP 10: Manual NPI remapping table (Jess mapping logic)
   - Remaps specific HCO NPIs/names to corrected "mapped" NPIs/names
   --------------------------------------------------------------------------- */
npi_mapping AS (
    SELECT current_npi, current_name, mapped_npi, mapped_name
    FROM (
        VALUES
        ('1144211301','Atrium Health Wake Forest Baptist Medical Center','1295789907','Atrium Health'),
        ('1184779332','Childrens Healthcare Of Atlanta Scottish Rite Hospital','1235339227','Emory University Hospital'),
        ('1851458038','Dr Patrick Leavey MD Office','1235582925','UT Health'),
        ('1225259039','Greenwood Genetics Center Inc.','1649261462','Greenwood Genetics Center Inc.'),
        ('1831318856','Greenwood Genetics Center Inc.','1649261462','Greenwood Genetics Center Inc.'),
        ('1356496772','Kaiser Permanente Fontana Medical Center','1013062769','Kaiser Permanente San Diego Medical Center'),
        ('1003947599','Univ. Pediatric Associates Inc.','1144266024','Indiana University Health'),
        ('1205822236','Yale Medicine','1013924182','Yale-New Haven Hospital')
    ) t (current_npi, current_name, mapped_npi, mapped_name)
),

/* ---------------------------------------------------------------------------
   STEP 11: Apply manual NPI + name overrides (Jess mapping)
   --------------------------------------------------------------------------- */
jess_file_implementation AS (
    SELECT
        * EXCEPT (a.hco_npi, a.hco_name),
        COALESCE(b.mapped_npi, a.hco_npi) AS hco_npi,
        COALESCE(b.mapped_name, a.hco_name) AS hco_name
    FROM finalizing_affiliations a
    LEFT JOIN npi_mapping b
        ON a.hco_npi = b.current_npi
),

/* ---------------------------------------------------------------------------
   STEP 12: Map secondary NPIs to primary NPIs (Julie mapping logic)
   - Produces active HCO NPI/name fields:
       hco_npi_active, hco_name_active
   - Flags if the original HCO NPI appeared as a secondary in Julie mapping table
   --------------------------------------------------------------------------- */
julie_file_mapping AS (
    SELECT
        a.*,
        COALESCE(b.primary_npi, a.hco_npi) AS hco_npi_active,
        CASE WHEN b.secondary_npi IS NOT NULL THEN 1 ELSE 0 END
            AS hco_npi_present_julies_file_flag,
        CASE
            WHEN b.primary_npi IS NOT NULL THEN b.primary_name
            ELSE a.hco_name
        END AS hco_name_active
    FROM jess_file_implementation a
    LEFT JOIN cmpa_insights_internal_schema.secondary_to_primary_npi b
        ON a.hco_npi = b.secondary_npi
),

/* ---------------------------------------------------------------------------
   STEP 13: Set HCO target flag
   - Target if:
     a) hco_npi_active is in the approved target list OR
     b) hco_npi_active is the mapped primary NPI for a target-listed secondary NPI
   --------------------------------------------------------------------------- */
updating_hco_target_flag AS (
    SELECT
        a.*,
        CASE
            WHEN (a.hco_npi_active IN ('1932280666','1912939703','1891765178','1861439952','1851458038',
                '1831318856','1760480503','1760476659','1750482022','1750458485',
                '1700128592','1689747552','1679973364','1669683512','1669462420',
                '1669429577','1659877280','1649347469','1649261462','1639370059',
                '1609824010','1598784555','1578693321','1568596765','1548212988',
                '1477643690','1477549756','1467525790','1447423959','1437365186',
                '1396882205','1376544320','1366556227','1366515488','1356496772',
                '1346297843','1336495910','1336245828','1326092404','1295789907',
                '1285832634','1285647933','1285174649','1275694184','1275564098',
                '1265694442','1235582925','1235339227','1235234535','1235214834',
                '1235148594','1225259039','1225249865','1215921457','1205935012',
                '1205822236','1194787218','1184779332','1184649345','1164686879',
                '1164426896','1154302727','1144548322','1144266024','1144211301',
                '1114969169','1114924834','1104819366','1093894131','1093808040',
                '1083949382','1083789630','1083630073','1073053757','1063702785',
                '1053632463','1043447253','1033439732','1023188851','1023105400',
                '1013924372','1013924182','1013143213','1013062769','1003961251',
                '1003947599','1003878539','1003102781','1003063280'
            ) OR a.hco_npi_active IN (
                SELECT DISTINCT primary_npi
                FROM com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi
                WHERE secondary_npi IN ('1932280666','1912939703','1891765178','1861439952','1851458038',
                    '1831318856','1760480503','1760476659','1750482022','1750458485',
                    '1700128592','1689747552','1679973364','1669683512','1669462420',
                    '1669429577','1659877280','1649347469','1649261462','1639370059',
                    '1609824010','1598784555','1578693321','1568596765','1548212988',
                    '1477643690','1477549756','1467525790','1447423959','1437365186',
                    '1396882205','1376544320','1366556227','1366515488','1356496772',
                    '1346297843','1336495910','1336245828','1326092404','1295789907',
                    '1285832634','1285647933','1285174649','1275694184','1275564098',
                    '1265694442','1235582925','1235339227','1235234535','1235214834',
                    '1235148594','1225259039','1225249865','1215921457','1205935012',
                    '1205822236','1194787218','1184779332','1184649345','1164686879',
                    '1164426896','1154302727','1144548322','1144266024','1144211301',
                    '1114969169','1114924834','1104819366','1093894131','1093808040',
                    '1083949382','1083789630','1083630073','1073053757','1063702785',
                    '1053632463','1043447253','1033439732','1023188851','1023105400',
                    '1013924372','1013924182','1013143213','1013062769','1003961251',
                    '1003947599','1003878539','1003102781','1003063280'
                )
            ))
            THEN a.hco_npi_active
            ELSE '-'
        END AS hco_target
    FROM julie_file_mapping a
),

/* ---------------------------------------------------------------------------
   STEP 14: Fetch latest HCO ZIP from VOD
   - Pull latest VALID address (status A/DS, verified not NS/U) for HCO entity
   --------------------------------------------------------------------------- */
hco_zip_v1 AS (
    SELECT DISTINCT
        a.npi_num__v AS hco_npi_active,
        b.address_line_1__v as hco_address,
        b.postal_code_cda__v AS hco_postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.npi_num__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_hco a
    JOIN com_raw.vod_address b
        ON b.entity_vid__v = a.vid__v
       AND b.entity_type__v = 'HCO'
       AND b.record_state__v = 'VALID'
       AND b.address_status__v IN ('A','DS')
       AND b.address_verification_status__v NOT IN ('NS','U')
    WHERE a.npi_num__v IN (
        SELECT DISTINCT hco_npi_active
        FROM updating_hco_target_flag
        WHERE hco_npi_active != '-'
    )
),

/* ---------------------------------------------------------------------------
   STEP 15: Select latest ZIP per HCO (rn=1)
   --------------------------------------------------------------------------- */
hco_zip_v2 AS (
    SELECT
        hco_npi_active,
        hco_address,
        hco_postal_code
    FROM hco_zip_v1
    WHERE rn = 1
),

/* ---------------------------------------------------------------------------
   STEP 16: Enrich / backfill HCO ZIP using:
   1) VOD latest ZIP (preferred)
   2) Komodo provider_zip (fallback)
   --------------------------------------------------------------------------- */
pulling_hco_zip_using_vod_komodo AS (
    SELECT
        a.* EXCEPT (hco_zip),
        case when b.hco_postal_code is not null then b.hco_address else c.provider_address end as hco_address,
        COALESCE(b.hco_postal_code, c.provider_zip, '-') AS hco_zip
    FROM updating_hco_target_flag a
    LEFT JOIN hco_zip_v2 b
        ON a.hco_npi_active = b.hco_npi_active
    LEFT JOIN com_raw.kom_providers c
        ON a.hco_npi_active = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ---------------------------------------------------------------------------
   STEP 17: Reassign territory & region using ZIP
   - Uses HCO ZIP if present; otherwise falls back to HCP ZIP
   --------------------------------------------------------------------------- */
territory_region_reassignment AS (
    SELECT
        a.* EXCEPT (territory, region),
        b.territory_id,
        b.territory_name AS territory,
        b.region_id,
        b.region_name AS region
    FROM pulling_hco_zip_using_vod_komodo a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON COALESCE(
               TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT),
               TRY_CAST(NULLIF(a.hcp_zipcode, '-') AS BIGINT)
           ) = b.zipcode
),

/* ---------------------------------------------------------------------------
   STEP 18: Final structural cleanup / de-dup
   --------------------------------------------------------------------------- */
final_output_v1 AS (
    SELECT DISTINCT
        hcp_npi,
        hcp_target,
        hcp_first_name,
        hcp_last_name,
        hcp_zipcode AS hcp_zip,
        hco_npi,
        hco_npi_active,
        hco_npi_present_julies_file_flag,
        hco_target,
        hco_name,
        hco_name_active,
        hco_address,
        hco_zip,
        territory_id,
        territory,
        region_id,
        region
    FROM territory_region_reassignment
),

/* ---------------------------------------------------------------------------
   STEP 19: Final enrichment with HCP specialty (Komodo INDIVIDUAL) + formatting
   --------------------------------------------------------------------------- */
final_output_v2 AS (
    SELECT DISTINCT
        hcp_npi,
        hcp_target,
        hcp_first_name,
        hcp_last_name,
        CONCAT(hcp_first_name, ' ', hcp_last_name) AS hcp_name,
        COALESCE(primary_specialty, '-') AS hcp_specialty,
        COALESCE(secondary_specialty, '-') AS hcp_secondary_specialty,
        COALESCE(hcp_zip, '-') AS hcp_zip,
        hco_npi as hco_npi_old,
        hco_name as hco_name_old,
        hco_npi_active AS hco_npi,
        hco_npi_present_julies_file_flag,
        hco_target,
        hco_name_active AS hco_name,
        coalesce(hco_address, '-') as hco_address,
        hco_zip,
        COALESCE(CAST(TRY_CAST(territory_id AS BIGINT) AS STRING), '-') AS territory_id,
        COALESCE(territory, '-') AS territory,
        COALESCE(CAST(TRY_CAST(region_id AS BIGINT) AS STRING), '-') AS region_id,
        COALESCE(region, '-') AS region
    FROM final_output_v1
    LEFT JOIN com_raw.kom_providers
        ON hcp_npi = npi
       AND provider_type = 'INDIVIDUAL'
)

SELECT *
FROM final_output_v2;


### Adding VOD/Komodo HCOs to the reference file

In [0]:
/* ============================================================================
   PURPOSE
   ----------------------------------------------------------------------------
   This script builds reference_file_v2 by:
   1. Starting from reference_file_v1
   2. Classifying HCPs into source lists (7541 / 6821 / Jess priority)
   3. Excluding specific geneticists from list logic
   4. Pulling HCO affiliations from Komodo
   5. Pulling most recent HCO affiliations from VOD
   6. Normalizing NPIs to primary NPIs
   7. Identifying the final HCO source (VOD / Komodo / Previous)
   ============================================================================ */

CREATE OR REPLACE TEMPORARY VIEW reference_file_v2 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Base input from reference_file_v1
   --------------------------------------------------------------------------- */
base_table AS (
    SELECT *
    FROM reference_file_v1
),

/* ---------------------------------------------------------------------------
   STEP 1: Load target HCP list (7541 list)
   --------------------------------------------------------------------------- */
target_hcp_list AS (
    SELECT DISTINCT CAST(hcp_npi AS STRING) AS hcp_npi
    FROM com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping
),

/* ---------------------------------------------------------------------------
   STEP 2: Hard-coded geneticist exclusion list
   --------------------------------------------------------------------------- */
geneticist_exclusion AS (
    SELECT '1528940079' AS hcp_npi UNION ALL
    SELECT '1609635499' UNION ALL
    SELECT '1114431707' UNION ALL
    SELECT '1619688116' UNION ALL
    SELECT '1740961929' UNION ALL
    SELECT '1073994448' UNION ALL
    SELECT '1538523675' UNION ALL
    SELECT '1396327128' UNION ALL
    SELECT '1447918628' UNION ALL
    SELECT '1336651918' UNION ALL
    SELECT '1467474502' UNION ALL
    SELECT '1518629294' UNION ALL
    SELECT '1407194038' UNION ALL
    SELECT '1104486638' UNION ALL
    SELECT '1922569649' UNION ALL
    SELECT '1003522897' UNION ALL
    SELECT '1821418765' UNION ALL
    SELECT '1063633311' UNION ALL
    SELECT '1689037434' UNION ALL
    SELECT '1669551321'
),

/* ---------------------------------------------------------------------------
   STEP 3: Assign source flag based on HCP presence and exclusions
   --------------------------------------------------------------------------- */
source_flag_addition AS (
    SELECT
        *,
        CASE
            WHEN hcp_npi IN (SELECT hcp_npi FROM target_hcp_list)
                THEN '7541 List'
            WHEN hcp_npi NOT IN (SELECT hcp_npi FROM target_hcp_list)
                 AND hcp_npi != '-'
                 AND hcp_npi NOT IN (SELECT hcp_npi FROM geneticist_exclusion)
                THEN '6821 List'
            ELSE 'Jess''s Priority List'
        END AS source_flag
    FROM base_table
),

/* ---------------------------------------------------------------------------
   STEP 4: Pull HCO affiliation from Komodo and map to primary NPI
   --------------------------------------------------------------------------- */
komodo_affiliation AS (
    SELECT
        a.*,
        COALESCE(c.primary_npi, b.hco_primary_npi, '-') AS komodo_hco
    FROM source_flag_addition a
    LEFT JOIN com_raw.kom_providers b
        ON a.hcp_npi = b.npi
       AND b.provider_type = 'INDIVIDUAL'
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi c
        ON b.hco_primary_npi = c.secondary_npi
),

/* ---------------------------------------------------------------------------
   STEP 5: Fetch HCP VID from VOD using HCP NPI
   --------------------------------------------------------------------------- */
hcp_vid AS (
    SELECT
        a.*,
        b.vid__v AS hcp_vid
    FROM komodo_affiliation a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

/* ---------------------------------------------------------------------------
   STEP 6: Pull and rank active VOD HCP–HCO affiliations by recency
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v1 AS (
    SELECT
        a.hcp_npi,
        c.vid__v AS hco_vid,
        c.corporate_name__v AS hco_name,
        c.npi_num__v AS hco_npi,
        b.modified_date__v,
        b.status_update_time__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

/* ---------------------------------------------------------------------------
   STEP 7: Select most recent VOD affiliation per HCP
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v2 AS (
    SELECT
        hcp_npi,
        hco_npi
    FROM ranked_vod_affiliations_v1
    WHERE rn = 1
      AND hco_npi IS NOT NULL
),

/* ---------------------------------------------------------------------------
   STEP 8: Normalize VOD HCO NPI to primary NPI
   --------------------------------------------------------------------------- */
vod_affiliations AS (
    SELECT
        a.*,
        COALESCE(c.primary_npi, b.hco_npi, '-') AS vod_hco
    FROM komodo_affiliation a
    LEFT JOIN ranked_vod_affiliations_v2 b
        ON a.hcp_npi = b.hcp_npi
    LEFT JOIN cmpa_insights_internal_schema.secondary_to_primary_npi c
        ON b.hco_npi = c.secondary_npi
),

/* ---------------------------------------------------------------------------
   STEP 9: Identify final HCO source (VOD / Komodo / Previous)
   --------------------------------------------------------------------------- */
is_hco_vod_komodo AS (
    SELECT
        a.*,
        CASE
            WHEN a.hco_npi = '-' THEN '-'
            WHEN a.hco_npi = a.vod_hco THEN 'vod'
            WHEN a.hco_npi = a.komodo_hco THEN 'komodo'
            ELSE 'Previous'
        END AS hco_source
    FROM vod_affiliations a
)

/* ---------------------------------------------------------------------------
   FINAL OUTPUT: Remove intermediate columns and expose source flag
   --------------------------------------------------------------------------- */
SELECT *
EXCEPT (komodo_hco, vod_hco)
FROM is_hco_vod_komodo;


### Fixing one HCO NPI to many name mapping

In [0]:
/* ============================================================================
   PURPOSE
   ----------------------------------------------------------------------------
   This script builds reference_file_v3 by:
   1. Starting from reference_file_v2
   2. Identifying HCO NPIs associated with multiple HCO names
   3. Standardizing HCO names using Komodo organization data
   4. Retaining existing names when Komodo name is unavailable
   ============================================================================ */

CREATE OR REPLACE TEMPORARY VIEW reference_file_v3 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Base input from reference_file_v2
   --------------------------------------------------------------------------- */
base_table AS (
    SELECT *
    FROM reference_file_v2
),

/* ---------------------------------------------------------------------------
   STEP 1: Identify HCO NPIs mapped to more than one HCO name
   --------------------------------------------------------------------------- */
hco_with_multiple_names AS (
    SELECT
        hco_npi
    FROM base_table
    GROUP BY hco_npi
    HAVING COUNT(DISTINCT hco_name) > 1
),

/* ---------------------------------------------------------------------------
   STEP 2: Standardize HCO names using Komodo organization name
   ---------------------------------------------------------------------------
   Logic:
   - Only applied to HCOs with multiple names
   - Prefer Komodo ORGANIZATION_NAME when available
   - Fall back to existing HCO name otherwise
   --------------------------------------------------------------------------- */
fixing_the_names AS (
    SELECT
        a.* EXCEPT (hco_name),
        COALESCE(b.ORGANIZATION_NAME, a.hco_name) AS hco_name
    FROM base_table a
    LEFT JOIN com_raw.kom_providers b
        ON a.hco_npi = b.npi
       AND b.PROVIDER_TYPE = 'ORGANIZATION'
       AND a.hco_npi IN (SELECT hco_npi FROM hco_with_multiple_names)
)

/* ---------------------------------------------------------------------------
   FINAL OUTPUT: Reference file with standardized HCO names
   --------------------------------------------------------------------------- */
SELECT hcp_npi, hcp_target, hcp_first_name, hcp_last_name, hcp_name, hcp_specialty, hcp_secondary_specialty, hcp_zip, hco_npi_old, hco_name_old, hco_npi, hco_name, hco_target, hco_npi_present_julies_file_flag, hco_address, hco_zip, territory_id, territory, region_id, region, source_flag, hco_source
FROM fixing_the_names;


### Fixing the issue of more than one hco_name_old for an hco_npi_old (Not used right now)

In [0]:
CREATE OR REPLACE TEMPORARY VIEW reference_file_v4 AS
WITH base_table AS (
  SELECT * FROM reference_file_v3
),
hco_with_multiple_names AS (
  SELECT hco_npi_old
  FROM base_table
  GROUP BY hco_npi_old
  HAVING COUNT(DISTINCT hco_name_old) > 1
),
fixing_hco_name_old AS (
  SELECT
    a.* EXCEPT(hco_name_old),
    COALESCE(b.ORGANIZATION_NAME, a.hco_name_old) AS hco_name_old
  FROM base_table a
  LEFT JOIN com_raw.kom_providers b
    ON a.hco_npi_old = b.npi
   AND b.PROVIDER_TYPE = 'ORGANIZATION'
   AND a.hco_npi_old IN (SELECT hco_npi_old FROM hco_with_multiple_names)
)
SELECT distinct hcp_npi, hcp_target, hcp_first_name, hcp_last_name, hcp_name, hcp_specialty, hcp_secondary_specialty, hcp_zip, hco_npi_old, hco_name_old, hco_npi, hco_name, hco_target, hco_npi_present_julies_file_flag, hco_zip, hco_address, territory_id, territory, region_id, region, source_flag, hco_source
FROM fixing_hco_name_old

In [0]:
create or replace temporary view reference_file_v5 as
with base_table as (
  select *
  from reference_file_v4
),
old_hco as (
  select distinct hco_npi_old, hco_name_old 
  from (select hco_npi_old, hco_name_old, row_number() over(partition by hco_npi_old order by hco_name_old asc) as rn
  from reference_file_v4
  where hco_npi_old != '-' and hco_name_old != '-')
  where rn = 1
),
new_hco as (
  select distinct hco_npi, hco_name
  from (select hco_npi, hco_name, row_number() over(partition by hco_npi order by hco_name asc) as rn
  from reference_file_v4
  where hco_npi != '-' and hco_name != '-')
  where rn = 1
),
final_name_mapping as (
  select a.* except(a.hco_name_old, a.hco_name), b.hco_name_old, c.hco_name
  from reference_file_v4 as a
  left join old_hco as b on a.hco_npi_old = b.hco_npi_old
  left join new_hco as c on a.hco_npi = c.hco_npi
)
select distinct hcp_npi, hcp_target, hcp_first_name, hcp_last_name, hcp_name, hcp_specialty, hcp_secondary_specialty, hcp_zip, hco_npi_old, hco_name_old, hco_npi, hco_name, hco_target, hco_npi_present_julies_file_flag, hco_zip, hco_address, territory_id, territory, region_id, region, source_flag, hco_source
from final_name_mapping

In [0]:
select count(*) from reference_file_v5

In [0]:
create or replace table com_edp_prd.cmpa_insights_internal_schema.reference_file as 
select * from reference_file_v5

In [0]:
select * 
from com_edp_prd.cmpa_insights_internal_schema.reference_file 

### QC

In [0]:
select distinct npi, ORGANIZATION_NAME
from com_raw.kom_providers
where HCO_PRIMARY_NPI in ('1205907763')

In [0]:
select distinct npi_num__v, corporate_name__v from com_raw.vod_hco
where npi_num__v in ('1194165589', '1093894131', '1265686000', '1013939750', '1255461935', '1124282652', '1184788945', '1225286172', '1750458485', '1326262742', '1447299425', '1013091727', '1306001409', '1235339227', '1093808040', '1316158793', '1003063280', '1003863168', '1841614963', '1326092404', '1205907763', '1023475845', '1225249865', '1922131903', '1013173970', '1699133199', '1053511444', '1013143213', '1023134657', '1083949382', '1073640744', '1306013552', '1275842007', '1235582925', '1154516243', '1013086941', '1730103565', '1053437871', '1285018218', '1649294026', '1053328948', '1083630073', '1578693321', '1811205545', '1033439732', '1063631943', '1043380637', '1639370059', '1124104005', '1336162874', '1053632463', '1043435902', '1326332289', '1043349905', '1336101534', '1114162013', '1568596765', '1043235880', '1285605444', '1285174649', '1154302727', '1053440701', '1295137404', '1598296410', '1285832634', '1033123666', '1144548322', '1023188851', '1124147277', '1023131174', '1659877280', '1083781892', '1063617280', '1487844015', '1528082971', '1174658470', '1629221411', '1043374713', '1053616375', '1043502750', '1669683512', '1073543799', '1477567857'
)

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file
where hcp_npi in ('1609003011','1215923115')

In [0]:
select * 
from com_edp_prd.com_intgr.distribution_sd_shipments